# A/B Testing, Rollbacks, and Audit Trails
Companion notebook for the **A/B Testing, Rollbacks, and Audit Trails** section of Chapter 9.

Ship prompt, tool, or graph changes gradually -- a canary slice or shadow mode first, then A/B on live traffic, with a fast rollback plan and an audit trail for every change.

The book's release config, as a real Python object:

In [1]:
release = {
    "release_id": "rel_2026_04_28_01",
    "change_type": "prompt_update",
    "from_version": "prompt_v12",
    "to_version": "prompt_v13",
    "traffic_split": {"A": 90, "B": 10},
    "rollback_rule": {
        "success_rate_drop": 0.03,   # ">3%"
        "policy_violation_increase": 0.01,  # ">1%"
    },
}
print(release)

{'release_id': 'rel_2026_04_28_01', 'change_type': 'prompt_update', 'from_version': 'prompt_v12', 'to_version': 'prompt_v13', 'traffic_split': {'A': 90, 'B': 10}, 'rollback_rule': {'success_rate_drop': 0.03, 'policy_violation_increase': 0.01}}


**Traffic split**: deterministic hash-based assignment, so the same user always lands in the same variant for the life of the release (no flip-flopping mid-session):

In [2]:
import hashlib


def assign_variant(user_id: str, traffic_split: dict, release_id: str) -> str:
    digest = hashlib.sha256(f"{release_id}:{user_id}".encode()).hexdigest()
    bucket = int(digest, 16) % 100
    threshold = traffic_split["A"]
    return "A" if bucket < threshold else "B"


sample_users = [f"user-{i}" for i in range(2000)]
assignments = [assign_variant(u, release["traffic_split"], release["release_id"]) for u in sample_users]
a_share = assignments.count("A") / len(assignments)
print(f"observed A share over {len(sample_users)} users: {a_share:.1%} (target: 90%)")

# Same user, same release -> same variant every time.
repeat = [assign_variant("user-42", release["traffic_split"], release["release_id"]) for _ in range(5)]
print("user-42 assigned consistently:", set(repeat))

observed A share over 2000 users: 89.8% (target: 90%)
user-42 assigned consistently: {'A'}


**A/B comparison**: measure more than final success -- latency, cost per successful task, and safety flags together, the same scorecard shape as notebook 02:

In [3]:
variant_metrics = {
    "A": {"success_rate": 0.90, "cost_per_success": 0.045, "policy_violation_rate": 0.001},
    "B": {"success_rate": 0.93, "cost_per_success": 0.061, "policy_violation_rate": 0.001},
}

for variant, m in variant_metrics.items():
    print(f"variant {variant}: success={m['success_rate']:.2%}  cost/success=${m['cost_per_success']:.3f}  policy_violations={m['policy_violation_rate']:.2%}")

print("\nB has higher success but also ~35% higher cost per success -- not an automatic win, exactly the trade-off the book warns about.")

variant A: success=90.00%  cost/success=$0.045  policy_violations=0.10%
variant B: success=93.00%  cost/success=$0.061  policy_violations=0.10%

B has higher success but also ~35% higher cost per success -- not an automatic win, exactly the trade-off the book warns about.


**Rollback rule**: compare the new version against the baseline, using the same thresholds recorded in `release`:

In [4]:
def should_rollback(baseline: dict, candidate: dict, rollback_rule: dict) -> dict:
    success_drop = baseline["success_rate"] - candidate["success_rate"]
    violation_increase = candidate["policy_violation_rate"] - baseline["policy_violation_rate"]

    reasons = []
    if success_drop > rollback_rule["success_rate_drop"]:
        reasons.append(f"success_rate dropped {success_drop:.2%} (limit {rollback_rule['success_rate_drop']:.0%})")
    if violation_increase > rollback_rule["policy_violation_increase"]:
        reasons.append(f"policy_violation_rate rose {violation_increase:.2%} (limit {rollback_rule['policy_violation_increase']:.0%})")

    return {"rollback": bool(reasons), "reasons": reasons}


# Variant B above: success improved and violations flat -> no rollback needed.
print(should_rollback(variant_metrics["A"], variant_metrics["B"], release["rollback_rule"]))

# A worse candidate: success drops sharply and violations rise -> should trigger rollback.
bad_candidate = {"success_rate": 0.83, "cost_per_success": 0.05, "policy_violation_rate": 0.02}
print(should_rollback(variant_metrics["A"], bad_candidate, release["rollback_rule"]))

{'rollback': False, 'reasons': []}
{'rollback': True, 'reasons': ['success_rate dropped 7.00% (limit 3%)', 'policy_violation_rate rose 1.90% (limit 1%)']}


**Audit trail**: who changed what, when, why, under which release ID -- with sensitive parameters hashed, not logged raw:

In [5]:
import hashlib
import time

audit_log = []


def record_audit_event(actor: str, action: str, release_id: str, reason: str, sensitive_params: dict | None = None):
    entry = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "actor": actor,
        "action": action,
        "release_id": release_id,
        "reason": reason,
    }
    if sensitive_params:
        entry["params_hash"] = hashlib.sha256(str(sorted(sensitive_params.items())).encode()).hexdigest()[:16]
    audit_log.append(entry)
    return entry


record_audit_event(
    actor="priya.k",
    action="deploy_prompt_update",
    release_id=release["release_id"],
    reason="reduce refund response length per support team feedback",
)
record_audit_event(
    actor="system",
    action="rollback_triggered",
    release_id=release["release_id"],
    reason="success_rate dropped past rollback_rule threshold",
    sensitive_params={"customer_id": "u-101", "order_id": "48321"},
)

for entry in audit_log:
    print(entry)

{'timestamp': '2026-08-30T07:38:45Z', 'actor': 'priya.k', 'action': 'deploy_prompt_update', 'release_id': 'rel_2026_04_28_01', 'reason': 'reduce refund response length per support team feedback'}
{'timestamp': '2026-08-30T07:38:45Z', 'actor': 'system', 'action': 'rollback_triggered', 'release_id': 'rel_2026_04_28_01', 'reason': 'success_rate dropped past rollback_rule threshold', 'params_hash': '9e6240c4bb57e164'}
